# Part 1: Data Ingestion

In [ ]:
#Imports
import requests
import polars as pl
from pathlib import Path
import plotly.express as px
import pandas as pd
import duckdb

In [ ]:
#Directory Set Up
RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data will be stored in: {RAW_DATA_DIR.resolve()}")

1. Programmatic Download 

In [ ]:
#Download required files using requests library
# File URLs
TRIP_DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

#Download Files
def download_file(url: str, save_path: Path):
        
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print(f"Saved to {save_path}")

download_file(TRIP_DATA_URL, RAW_DATA_DIR / "yellow_tripdata_2024-01.parquet")
download_file(ZONE_LOOKUP_URL, RAW_DATA_DIR / "taxi_zone_lookup.csv")


In [ ]:
#Load Polars 
trip_df = pl.read_parquet(RAW_DATA_DIR / "yellow_tripdata_2024-01.parquet")
zone_df = pl.read_csv(RAW_DATA_DIR / "taxi_zone_lookup.csv")

In [ ]:
# Columns used
EXPECTED_COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type"
]

2. Data Validation

In [ ]:
def validate_trip_data(df: pl.DataFrame):
    print("Running validation checks...")
    
    #  a. Verify all expected columns exist in the datase
    expected = set(EXPECTED_COLUMNS)
    actual = set(trip_df.columns)

    if not expected.issubset(actual):
        missing = expected - actual
        raise ValueError(f"Missing columns: {missing}")

    print("Column validation passed")
    
    #  b. Check that date columns are valid datetime types
    datetime_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

    for col in datetime_cols:
        if trip_df.schema[col] != pl.Datetime:
            raise TypeError(f"{col} is not Datetime type")

    print("Datetime validation passed")

    # c. Report total row count and print a summary to the console
    print("Validation Passed")
    print(f"Total rows: {df.height:,}")
    if trip_df.height == 0:
        raise ValueError("Validation failed. Dataset is empty.")
    print(df.describe())

validate_trip_data(trip_df)


The validation checks confirm that the dataset contains approximately 2.96 million records with correctly formatted datetime columns spanning January 2024. However, summary statistics reveal the presence of extreme fare outliers, including negative values and fares exceeding $5000, which are not realistic for taxi trips. These findings justify the data cleaning steps implemented in Part 2 to remove invalid trips and ensure analytical reliability. Overall, the schema and row count validation indicate the dataset loaded correctly and is structurally sound.

# Part 2: Data Transformation & Analysis

Data Cleaning

In [ ]:
CRITICAL_NULL_COLS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
]

def clean_trips(df: pl.DataFrame) -> tuple[pl.DataFrame, dict]:
    counts = {"Starting_rows": df.height}

    # e) Drop nulls in critical columns
    df1 = df.drop_nulls(subset=CRITICAL_NULL_COLS)
    counts["Removed_null_critical"] = counts["Starting_rows"] - df1.height

    # f) Filter invalid trips:
    df2 = df1.filter(
        (pl.col("trip_distance") > 0)
        & (pl.col("fare_amount") >= 0)
        & (pl.col("fare_amount") <= 500)
    )
    counts["Removed_invalid_distance_or_fare"] = df1.height - df2.height

    # g) Remove trips where dropoff < pickup
    df3 = df2.filter(pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"))
    counts["Removed_dropoff_before_pickup"] = df2.height - df3.height

    counts["Final_rows"] = df3.height
    return df3, counts

clean_df, clean_counts = clean_trips(trip_df)

print("Summary of cleaning")
for k, v in clean_counts.items():
    print(f"{k}: {v:,}")


The cleaning process removed approximately 94,522 records (3.2% of the dataset), primarily due to invalid fare or distance values, while only 56 records contained logically inconsistent timestamps. No critical fields contained null values, indicating strong structural integrity of the raw dataset. The final dataset retains over 96% of original records, ensuring that subsequent analysis remains representative while eliminating anomalies that could distort statistical results.

Feature Engineering

In [ ]:
def add_features(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        # i) trip_duration_minutes
        ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
         .dt.total_seconds() / 60
        ).alias("trip_duration_minutes"),

        # k) pickup_hour
        pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),

        # l) pickup_day_of_week (day name)
        pl.col("tpep_pickup_datetime").dt.strftime("%A").alias("pickup_day_of_week"),
    )

    # j) trip_speed_mph = distance / (duration_hours), handle division by zero
    df = df.with_columns(
        pl.when(pl.col("trip_duration_minutes") > 0)
          .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
          .otherwise(None)
          .alias("trip_speed_mph")
    )

    return df

feat_df = add_features(clean_df)

new_cols = {"trip_duration_minutes", "trip_speed_mph", "pickup_hour", "pickup_day_of_week"}
print("New columns present:", new_cols.issubset(set(feat_df.columns)))

feat_df.select([
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "trip_duration_minutes",
    "trip_speed_mph",
    "pickup_hour",
    "pickup_day_of_week"
]).head(10)

The engineered features appear internally consistent and logically valid. Trip durations are positive and fall within realistic urban travel ranges, while calculated speeds range between approximately 2–25 mph, consistent with typical NYC driving conditions. The extracted pickup hour and day-of-week values correctly correspond to the original timestamps (e.g., January 1, 2024 correctly identified as Monday at hour 0). These observations confirm that feature engineering was implemented correctly and aligns with expected real-world behavior.

SQL Analysis

In [ ]:
#Load data into duckdb for SQL analysis
con = duckdb.connect()

con.register("trips", feat_df)
con.register("zones", zone_df)

con.execute("SELECT COUNT(*) AS n FROM trips").df().head()


In [ ]:
# m) Top 10 busiest pickup zones by total number of trips
q1 = """
SELECT
  z.Zone AS pickup_zone,
  COUNT(*) AS trip_count
FROM trips t
JOIN zones z
  ON t.PULocationID = CAST(z.LocationID AS INTEGER)
GROUP BY 1
ORDER BY trip_count DESC
LIMIT 10;
"""
con.execute(q1).df()


 Taxi demand is heavily concentrated in Midtown Manhattan and affluent residential areas such as the Upper East and Upper West Sides. The presence of JFK and LaGuardia airports among the top pickup locations highlights the significant role of airport travel in total trip volume. These results indicate that commercial hubs, tourist districts, and major transportation nodes drive the majority of taxi demand in January 2024.

In [ ]:
# n. Average fare amount for each hour of the day
q2 = """
SELECT
  pickup_hour,
  AVG(fare_amount) AS avg_fare_amount
FROM trips
GROUP BY pickup_hour
ORDER BY pickup_hour;
"""
con.execute(q2).df()


Average fares are highest during the early morning hours (4–5 AM), peaking at over $27 at 5 AM. This likely reflects longer airport trips or lower traffic conditions enabling longer-distance travel. During midday and typical commute hours, average fares stabilize around $17–19, suggesting shorter urban trips dominate those periods.

In [ ]:
# o. Percentage of trips use each payment type
q3 = """
SELECT
  CASE payment_type
    WHEN 1 THEN 'Credit Card'
    WHEN 2 THEN 'Cash'
    WHEN 3 THEN 'No Charge'
    WHEN 4 THEN 'Dispute'
    WHEN 5 THEN 'Unknown'
    ELSE 'Other'
  END AS payment_type_label,
  COUNT(*) AS trips,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_trips
FROM trips
GROUP BY payment_type
ORDER BY trips DESC;
"""
con.execute(q3).df()


Credit card payments dominate NYC taxi trips, accounting for over 80% of all transactions. Cash payments represent only about 15%, while disputes and no-charge trips are extremely rare. This strong shift toward electronic payments suggests high digital adoption and implies that tip data (which is auto-recorded for card payments) likely captures the majority of tipping behavior.

### SQL Query 4: Average tip percentage by day of week (credit card only)

In [ ]:
# p. Average tip percentage (tip_amount / fare_amount) for each day of the week, only for cash payments (payment_type = 1) and fare_amount > 0
q4 = """
SELECT
  pickup_day_of_week,
  AVG( (tip_amount / fare_amount) * 100.0 ) AS avg_tip_pct
FROM trips
WHERE payment_type = 1
  AND fare_amount > 0
GROUP BY pickup_day_of_week
ORDER BY
  CASE pickup_day_of_week
    WHEN 'Monday' THEN 1
    WHEN 'Tuesday' THEN 2
    WHEN 'Wednesday' THEN 3
    WHEN 'Thursday' THEN 4
    WHEN 'Friday' THEN 5
    WHEN 'Saturday' THEN 6
    WHEN 'Sunday' THEN 7
    ELSE 8
  END;
"""
con.execute(q4).df()


Tip percentages remain relatively stable around 25–26% for most days of the week, indicating consistent tipping behavior. However, Thursday stands out with a significantly higher average tip rate of nearly 30%. This spike may reflect increased social or pre-weekend travel, where riders are more likely to tip generously compared to routine weekday commuting trips.

In [ ]:
# q. Top 5 most common pickup-dropoff zone pairs (most frequent routes)
q5 = """
SELECT
  pu.Zone AS pickup_zone,
  dz.Zone AS dropoff_zone,
  COUNT(*) AS trip_count
FROM trips t
JOIN zones pu ON t.PULocationID = pu.LocationID
JOIN zones dz ON t.DOLocationID = dz.LocationID
GROUP BY 1, 2
ORDER BY trip_count DESC
LIMIT 5;
"""
con.execute(q5).df()


The most common taxi flows occur within and between Upper East Side zones, indicating strong localized travel patterns in this residential area. The bidirectional dominance between Upper East Side North and South suggests frequent short-distance neighborhood trips. Additionally, the Midtown Center to Upper East Side route highlights movement between major commercial and residential districts, reflecting commuter or business-related travel patterns.

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

feat_df.write_parquet(PROCESSED_DIR / "clean_trips.parquet")

print("Processed dataset saved successfully ✅")


# Part 3: Dashboard Development

In [ ]:
TRIPS_PATH = Path("data/processed/clean_trips.parquet")
ZONES_PATH = Path("data/raw/taxi_zone_lookup.csv")

feat_df = pl.read_parquet(TRIPS_PATH)
zone_df = pl.read_csv(ZONES_PATH)

# Small lookup table for joins
zones_small = zone_df.select(["LocationID", "Zone"])


In [ ]:
# Add pickup_date for date filtering
feat_df = feat_df.with_columns(
    pl.col("tpep_pickup_datetime").dt.date().alias("pickup_date")
)

# Pick a filter window
start_date = feat_df.select(pl.col("pickup_date").min()).item()
end_date   = feat_df.select(pl.col("pickup_date").max()).item()

hour_min, hour_max = 0, 23
payment_types = [1, 2, 3, 4, 5]  # 1=card, 2=cash, 3=no charge, 4=dispute, 5=unknown

filtered = feat_df.filter(
    (pl.col("pickup_date") >= pl.lit(start_date)) &
    (pl.col("pickup_date") <= pl.lit(end_date)) &
    (pl.col("pickup_hour") >= hour_min) &
    (pl.col("pickup_hour") <= hour_max) &
    (pl.col("payment_type").is_in(payment_types))
)

print(f"Filtered rows: {filtered.height:,}")


In [ ]:
total_trips = filtered.height
avg_fare = filtered.select(pl.col("fare_amount").mean()).item()
total_revenue = filtered.select(pl.col("total_amount").sum()).item()
avg_distance = filtered.select(pl.col("trip_distance").mean()).item()
avg_duration = filtered.select(pl.col("trip_duration_minutes").mean()).item()

print("=== Key Metrics (Prototype) ===")
print(f"Total trips: {total_trips:,}")
print(f"Average fare: ${avg_fare:,.2f}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Average distance: {avg_distance:,.2f} miles")
print(f"Average duration: {avg_duration:,.2f} minutes")


Required Visualizations

In [ ]:
# r. Bar Chart with top 10 pickup zones by trip count 
top_pu = (
    filtered.join(zones_small, left_on="PULocationID", right_on="LocationID", how="left")
    .group_by("Zone")
    .agg(pl.len().alias("trip_count"))
    .sort("trip_count", descending=True)
    .head(10)
)

fig1 = px.bar(top_pu.to_pandas(), x="Zone", y="trip_count",
              title="Top 10 Pickup Zones by Trip Count")
fig1.update_layout(xaxis_title="Pickup Zone", yaxis_title="Trips", xaxis_tickangle=-35)
fig1.show()


In [ ]:
# s. Line Chart showing average fare amount for each hour of the day
fare_by_hour = (
    filtered.group_by("pickup_hour")
    .agg(pl.col("fare_amount").mean().alias("avg_fare"))
    .sort("pickup_hour")
)

fig2 = px.line(fare_by_hour.to_pandas(), x="pickup_hour", y="avg_fare", markers=True,
               title="Average Fare Amount by Pickup Hour")
fig2.update_layout(xaxis_title="Pickup Hour (0–23)", yaxis_title="Average Fare ($)")
fig2.show()


In [ ]:
# t. Histogram of trip distances 
dist_pd = filtered.select(pl.col("trip_distance").clip(0, 30).alias("trip_distance_capped")).to_pandas()

fig3 = px.histogram(dist_pd, x="trip_distance_capped", nbins=40,
                    title="Trip Distance Distribution (Capped at 30 miles)")
fig3.update_layout(xaxis_title="Trip Distance (miles)", yaxis_title="Trips")
fig3.show()


In [ ]:
# u. Pie Chart showing the percentage breakdown of payment types
payment_map = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
}

pay_counts = (
    filtered.group_by("payment_type")
    .agg(pl.len().alias("trips"))
    .with_columns(
        pl.col("payment_type")
        .map_elements(lambda x: payment_map.get(x, "Other"), return_dtype=pl.Utf8)
        .alias("payment_label")
    )
    .sort("trips", descending=True)
)
fig4 = px.pie(pay_counts.to_pandas(), names="payment_label", values="trips",
              title="Payment Type Breakdown")
fig4.show()


In [ ]:
# v. Heatmap showing the number of trips for each combination of pickup day of week and pickup hour
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

heat = (
    filtered.group_by(["pickup_day_of_week", "pickup_hour"])
    .agg(pl.len().alias("trips"))
)

heat_pd = heat.to_pandas()
heat_pd["pickup_day_of_week"] = pd.Categorical(
    heat_pd["pickup_day_of_week"], categories=dow_order, ordered=True
)
heat_pd = heat_pd.sort_values(["pickup_day_of_week", "pickup_hour"])

fig5 = px.density_heatmap(
    heat_pd,
    x="pickup_hour",
    y="pickup_day_of_week",
    z="trips",
    title="Trips by Day of Week and Hour"
)
fig5.update_layout(xaxis_title="Hour (0–23)", yaxis_title="Day of Week")
fig5.show()


## AI TOOLS
### Chat GPT
Used for questions, research and debugging.